# FAIR Organelle Segmentation pipeline

In [ ]:
import sys
sys.path.append('..')

# Relies on https://github.com/volume-em/empanada-napari.git@inf_pipeline_dev
from empanada_napari._volume_inference import VolumeInferenceWidget
import glob
import matplotlib.pyplot as plt
import numpy as np
import ome_zarr
from skimage.transform import resize

from src.fair_segmentation.image_util import *

## Load source data

In [ ]:
source_path = 'D:/slides/AMC_EM/**/*.tif'

filenames = glob.glob(source_path)
filenames


## Process data

In [ ]:
downscale = 2

datas = []
for filename in filenames:
    data, metadata, pixel_size = extract_tiff_olympus(filename)
    new_shape = list(data.shape)
    new_shape[0] //= downscale
    new_shape[1] //= downscale
    data = resize(data, new_shape)
    datas.append(float2int_image(norm_image_variance2(data)))

## Select data

In [ ]:
%matplotlib inline

data = datas[0]

plt.imshow(data)

## Run model

In [ ]:
inference_plane = "xy"

if data.ndim == 2:
    data = np.expand_dims(data, 0)

inference_config = VolumeInferenceWidget(viewer=None,
                                        image_layer=data,
                                        return_panoptic=True,
                                        inference_plane=inference_plane,
                                        model_config='MitoNet_v1_mini',
                                        use_gpu=True)

stack, axis_name, trackers_dict = inference_config.config_and_run_inference(use_thread=False)

## Show output

In [ ]:
%matplotlib inline
print(np.max(stack))
plt.imshow(stack[0])